# 02 - ETL Pipeline: NASA NeoWs → Supabase

Pipeline de extracción, transformación y carga de datos de asteroides
desde la API pública de NASA hacia Supabase (PostgreSQL).

**Fuente:** NASA NeoWs API
**Destino:** Supabase (PostgreSQL)
**Período:** Enero 2021 → Abril 2026

## 1. Configuración e imports

In [ ]:
import requests
import pandas as pd
from datetime import datetime, timedelta
from dotenv import load_dotenv
from supabase import create_client
import os
import time

load_dotenv(dotenv_path='../.env')
API_KEY = os.getenv("NASA_API_KEY")

# Verificar que la key cargó bien
print("API Key cargada:", API_KEY is not None)

## 2. Funciones de extracción y transformación

`get_asteroids()` — wrapper de la API con paginación semanal.
La API acepta un máximo de 7 días por request, por lo que se itera
en ventanas de 7 días sobre el período completo.

`extraer_campos()` — normaliza el response JSON y selecciona
únicamente los campos relevantes para el modelo, todos en kilómetros.

In [ ]:
def get_asteroids(start_date, end_date):
    url = "https://api.nasa.gov/neo/rest/v1/feed"
    params = {"start_date": start_date, "end_date": end_date, "api_key": API_KEY}
    response = requests.get(url, params=params)
    return response.json()['near_earth_objects']

def extraer_campos(asteroide):
    close_approach = asteroide['close_approach_data'][0]
    return {
        'id': asteroide['id'],
        'name': asteroide['name'],
        'absolute_magnitude_h': asteroide['absolute_magnitude_h'],
        'diameter_min_km': asteroide['estimated_diameter']['kilometers']['estimated_diameter_min'],
        'diameter_max_km': asteroide['estimated_diameter']['kilometers']['estimated_diameter_max'],
        'velocity_km_per_hour': float(close_approach['relative_velocity']['kilometers_per_hour']),
        'miss_distance_km': float(close_approach['miss_distance']['kilometers']),
        'close_approach_date': close_approach['close_approach_date'],
        'is_potentially_hazardous': asteroide['is_potentially_hazardous_asteroid']
    }

## 3. Descarga, deduplicación y carga

**Decisión de diseño: deduplicación por asteroide:**
La API organiza los datos por fecha de close approach. Un mismo asteroide
puede aparecer múltiples veces si tuvo varios acercamientos en el período.

La unidad de análisis del modelo es el asteroide, no el evento de
acercamiento. `is_potentially_hazardous` es una propiedad fija del objeto
orbital, no varía entre acercamientos. Se conserva el primer registro
por asteroide (`drop_duplicates(subset='id', keep='first')`).

En un sistema productivo, los eventos de acercamiento vivirían en una
tabla separada con FK → asteroides. Esta simplificación está justificada
para el alcance del proyecto.

In [ ]:
load_dotenv(dotenv_path='../.env')

SUPABASE_URL = os.getenv("SUPABASE_URL")
SUPABASE_KEY = os.getenv("SUPABASE_KEY")

supabase = create_client(SUPABASE_URL, SUPABASE_KEY)

start = datetime(2021, 1, 1)
end = datetime(2026, 3, 31)
todos_los_asteroides = []
current = start

while current < end:
    siguiente = min(current + timedelta(days=7), end)
    datos = get_asteroids(current.strftime('%Y-%m-%d'), siguiente.strftime('%Y-%m-%d'))
    for fecha, asteroides in datos.items():
        for asteroide in asteroides:
            todos_los_asteroides.append(extraer_campos(asteroide))
    print(f"✓ {current.strftime('%Y-%m-%d')} — acumulados: {len(todos_los_asteroides)}")
    current = siguiente + timedelta(days=1)
    time.sleep(0.5)

df = pd.DataFrame(todos_los_asteroides)

# Cuántos registros teníamos
print("Antes de deduplicar:", len(df))

# Nos quedamos con el primer acercamiento de cada asteroide
df = df.drop_duplicates(subset='id', keep='first')

# Cuántos quedaron
print("Después de deduplicar:", len(df))

print("\nCargando en Supabase...")
batch_size = 500
for i in range(0, len(df), batch_size):
    batch = df.iloc[i:i+batch_size].to_dict(orient='records')
    supabase.table('asteroids').upsert(batch).execute()
    print(f"  ✓ Insertados registros {i} a {i+len(batch)}")

print("\nListo")

In [ ]:
df.to_csv('../data/asteroids_raw.csv', index=False)
print("CSV guardado en data/asteroids_raw.csv")

## Nota sobre reproducibilidad

La API de NASA actualiza retroactivamente sus registros históricos.
Para garantizar resultados reproducibles, el dataset se guardó como
CSV local en data/asteroids_raw.csv después de la primera extracción.

Cualquier re-ejecución del pipeline debe cargar desde ese CSV
en lugar de llamar a la API nuevamente.